# Déchets Plastiques mal gérés et mortalité Infantile (2019)

**Groupe :** GOMA Rayan, PRIBETICH Marius, LUCIBELLO Anton

> En 2019, parmi les pays et territoires disposant des données, observe-t-on une association descriptive entre les déchets plastiques mal gérés par habitant et le taux de mortalité infantile ? Cette association paraît-elle similaire lorsqu’on compare séparément les groupes de PIB par habitant et d’accès à l’assainissement de base ?

Cette analyse compare des indicateurs nationaux. Elle ne démontre pas que le plastique cause des décès et ne décrit pas l’exposition d’un enfant individuellement.

## 1. Sources, variables et méthode

- **Déchets plastiques mal gérés :** Our World in Data, source Meijer et al. (2021), en kg par personne et par an. L’indicateur ne mesure pas directement l’exposition des personnes.
- **Mortalité infantile :** World Bank WDI `SP.DYN.IMRT.IN`, décès avant un an pour 1 000 naissances vivantes.
- **PIB par habitant :** World Bank WDI `NY.GDP.PCAP.PP.KD`, dollars internationaux constants de 2021 en PPA.
- **Assainissement :** World Bank WDI `SH.STA.BASS.ZS`, part de la population ayant accès à au moins un service d’assainissement de base. Ce n’est pas une mesure de la collecte des déchets plastiques.

On filtre les agrégats, on garde l’année 2019, puis on joint les tables par code géographique ISO3 et année. Les données manquantes ne sont pas remplacées par zéro. Le PIB et l’assainissement sont comparés séparément comme contexte; ils ne sont pas présentés comme des contrôles causaux.

## 2. Chargement des bibliothèques et fonctions du projet

`Path` construit des chemins compatibles avec Windows, macOS et Linux. Si le notebook est exécuté seul, il télécharge automatiquement le dépôt du projet pour récupérer le module `prepare_data.py` et la structure des dossiers. `pandas` sert à lire et résumer les tableaux ; `matplotlib` sert à tracer les graphiques.

In [9]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display


PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "cleaning.py").exists():
    if (PROJECT_DIR.parent / "cleaning.py").exists():
        PROJECT_DIR = PROJECT_DIR.parent
    else:
        repo_dir = PROJECT_DIR / "Python-pour-la-data-science"
        if not repo_dir.exists():
            subprocess.run(
                ["git", "clone", "https://github.com/Rayanisworking/Python-pour-la-data-science.git"],
                check=True,
            )
        PROJECT_DIR = repo_dir

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from download import download, RAW_ROOT
from cleaning import prepare_project_data


FIGURES_DIR = PROJECT_DIR / "outputs" / "figures"
TABLES_DIR = PROJECT_DIR / "outputs" / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dossier du projet prêt : {PROJECT_DIR}")

Dossier du projet prêt : /home/onyxia/work/Python-pour-la-data-science


## 3. Préparer les tables et vérifier la jointure

Les cellules ci-dessous vérifient d'abord la présence des données brutes dans `data/raw/` (et lancent automatiquement le téléchargement officiel via `download()` si elles sont absentes). La fonction `prepare_project_data()` filtre ensuite les agrégats régionaux, conserve l'année 2019, joint les indicateurs par code pays ISO3 et sauvegarde les tables nettoyées dans `data/cleaned/`.

In [10]:

required_raw_files = [
    RAW_ROOT / "owid_mismanaged_plastic_per_capita.csv",
    RAW_ROOT / "wdi_countries.json",
    RAW_ROOT / "wdi_infant_mortality_2019.json",
    RAW_ROOT / "wdi_gdp_per_capita_ppp_2019.json",
    RAW_ROOT / "wdi_basic_sanitation_2019.json",
]

if not all(f.exists() for f in required_raw_files):
    print("Données brutes manquantes : téléchargement automatique en cours...")
    download()


tables = prepare_project_data()

df_merged = tables["merged"]
df_pair = tables["pair_plastic_infant"]
df_complete = tables["complete"]
df_excluded = tables["excluded"]
country_metadata = tables["country_metadata"]

print(f"Pays/territoires appariés (OWID 2019) : {len(df_merged)}")
print(f"Pays/territoires avec déchets et mortalité : {len(df_pair)}")
print(f"Pays/territoires complets pour les quatre variables : {len(df_complete)}")

Pays/territoires appariés (OWID 2019) : 153
Pays/territoires avec déchets et mortalité : 151
Pays/territoires complets pour les quatre variables : 139


### Bilan de la jointure

Un effectif différent selon les colonnes signifie que certaines sources n'ont pas de valeur renseignée pour certains pays ou territoires. On affiche ci-dessous les effectifs réels à chaque étape du filtrage plutôt que de supprimer silencieusement les données manquantes. Les cas complets servent aux comparaisons par PIB et assainissement; l’association principale utilise tous les cas disponibles pour les deux variables concernées.

In [11]:

audit_df = pd.DataFrame([
    {"controle": "Entités OWID 2019 retenues (pays/territoires)", "effectif": len(df_merged)},
    {"controle": "Entités OWID 2019 exclues (agrégats/hors périmètre)", "effectif": len(df_excluded)},
    {"controle": "Pays avec déchets + mortalité", "effectif": len(df_pair)},
    {"controle": "Cas complets (4 variables)", "effectif": len(df_complete)},
])
display(audit_df)


cols = ["plastic_kg_per_person", "infant_mortality_per_1000", "gdp_pc_ppp_2019", "basic_sanitation_pct"]
missing_df = pd.DataFrame({
    "variable": cols,
    "valeurs_presentes": [df_merged[c].notna().sum() for c in cols],
    "valeurs_manquantes": [df_merged[c].isna().sum() for c in cols],
})
display(missing_df)


range_df = pd.DataFrame({
    "variable": cols,
    "min": [df_merged[c].min() for c in cols],
    "max": [df_merged[c].max() for c in cols],
})
display(range_df)

print("Exemples d'entités OWID exclues du périmètre World Bank :")
display(tables["excluded"].head(10))

,controle,effectif
0,Entités OWID 2019 retenues (pays/territoires),153
1,Entités OWID 2019 exclues (agrégats/hors périm...,12
2,Pays avec déchets + mortalité,151
3,Cas complets (4 variables),139


,variable,valeurs_presentes,valeurs_manquantes
0,plastic_kg_per_person,153,0
1,infant_mortality_per_1000,151,2
2,gdp_pc_ppp_2019,148,5
3,basic_sanitation_pct,145,8


,variable,min,max
0,plastic_kg_per_person,0.002720,69.515860
1,infant_mortality_per_1000,1.800000,75.000000
2,gdp_pc_ppp_2019,1394.764615,120678.341979
3,basic_sanitation_pct,13.169338,100.000000


Exemples d'entités OWID exclues du périmètre World Bank :


,entity,iso3,year,plastic_kg_per_person
0,Africa,OWID_AFR,2019,10.465928
6,Asia,OWID_ASI,2019,8.667235
45,Europe,OWID_EUR,2019,1.580226
49,French Guiana,GUF,2019,0.432990
57,Guadeloupe,GLP,2019,0.405000
91,Martinique,MTQ,2019,0.369681
105,North America,OWID_NAM,2019,5.257716
108,Oceania,OWID_OCE,2019,3.240268
121,Reunion,REU,2019,0.262092
146,Taiwan,TWN,2019,0.315555


## 4. Décrire les quatre variables

La médiane et les quartiles décrivent le centre et la dispersion sans être autant tirés par les valeurs extrêmes que la moyenne. `n` indique combien de valeurs sont effectivement disponibles pour chaque variable.

In [12]:
variable_info = {
    "plastic_kg_per_person": "Déchets plastiques mal gérés (kg/personne/an)",
    "infant_mortality_per_1000": "Mortalité infantile (décès pour 1 000 naissances)",
    "gdp_pc_ppp_2019": "PIB/habitant (dollars internationaux PPA 2021)",
    "basic_sanitation_pct": "Assainissement de base (% de la population)",
}

summary_rows = []
for column, label in variable_info.items():
    values = df_merged[column].dropna()
    summary_rows.append({
        "variable": label,
        "n": len(values),
        "manquants": int(df_merged[column].isna().sum()),
        "mediane": values.median(),
        "Q1": values.quantile(0.25),
        "Q3": values.quantile(0.75),
        "minimum": values.min(),
        "maximum": values.max(),
    })

df_descriptives = pd.DataFrame(summary_rows)
display(df_descriptives.round(2))
df_descriptives.to_csv(TABLES_DIR / "statistiques_descriptives.csv", index=False)

,variable,n,manquants,mediane,Q1,Q3,minimum,maximum
0,Déchets plastiques mal gérés (kg/personne/an),153,0,6.26,0.54,14.30,0.00,69.52
1,Mortalité infantile (décès pour 1 000 naissances),151,2,13.20,5.55,30.35,1.80,75.00
2,PIB/habitant (dollars internationaux PPA 2021),148,5,17478.10,6708.19,37534.90,1394.76,120678.34
3,Assainissement de base (% de la population),145,8,91.50,66.04,98.65,13.17,100.00
